In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Colab: one-time installs (paste + run)
!apt-get update -y
!apt-get install -y ffmpeg -qq
# Python packages
!pip install -q git+https://github.com/openai/whisper.git
!pip install -q transformers soundfile librosa scikit-learn joblib pandas


Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
  Installing b

In [ ]:
# Training notebook: Tone model (Wav2Vec2 embeddings -> sklearn classifier)
# Copy and run this whole Python cell in Colab

import os, glob, re, math, json
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
import joblib

# HuggingFace
import torch
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model

# sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ------------------ CONFIG ------------------
TRAIN_AUDIO_DIR = "/content/drive/MyDrive/Hackthon/INNOV8 3.0/INNOV8 3.0"   # folder with your 5 training audios / labeled audios
OUTPUT_DIR = "/content/drive/MyDrive/Hackthon/INNOV8_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# CSV with columns: filename,label OR fallback to filenames with keywords (whisper/emotional/neutral)
LABEL_CSV = os.path.join(TRAIN_AUDIO_DIR, "train_labels.csv")  # optional: if exists, will be used

# Wav2Vec2 model for embeddings (small)
HF_WAV2VEC = "facebook/wav2vec2-base-960h"   # small-ish, good embeddings

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ------------------ Load feature extractor & model ------------------
print("Loading Wav2Vec2 feature extractor & model (this may take a moment)...")
feat_ext = Wav2Vec2FeatureExtractor.from_pretrained(HF_WAV2VEC)
wv_model = Wav2Vec2Model.from_pretrained(HF_WAV2VEC).to(device)
wv_model.eval()

# ------------------ Helper: get embedding for a file ------------------
import soundfile as sf
import librosa

def get_embedding(file_path, sr=16000):
    # load audio
    wav, _ = librosa.load(file_path, sr=sr)
    # Wav2Vec2 expects 1D float32 array
    inputs = feat_ext(wav, sampling_rate=sr, return_tensors="pt", padding=True)
    with torch.no_grad():
        input_values = inputs.input_values.to(device)  # shape (1, time)
        outputs = wv_model(input_values)
        # outputs.last_hidden_state -> (1, seq_len, hidden)
        emb = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()  # mean pooling over time
    return emb  # 1D numpy array

# ------------------ Build dataset (labels) ------------------
audio_files = sorted(glob.glob(os.path.join(TRAIN_AUDIO_DIR, "*.mp3")) + glob.glob(os.path.join(TRAIN_AUDIO_DIR, "*.wav")))
if not audio_files:
    raise SystemExit(f"No audio files found in {TRAIN_AUDIO_DIR} - put your labeled training audio there.")

df = []
if os.path.exists(LABEL_CSV):
    print("Loading label CSV:", LABEL_CSV)
    labdf = pd.read_csv(LABEL_CSV)
    # Expect columns 'filename' (or 'path') and 'label'
    for _, row in labdf.iterrows():
        fn = row.get("filename") or row.get("path") or row.get("file")
        label = row.get("label")
        if not fn or not label:
            continue
        # support relative filename -> full path
        if not os.path.isabs(fn):
            full = os.path.join(TRAIN_AUDIO_DIR, fn)
        else:
            full = fn
        if os.path.exists(full):
            df.append((full, str(label).strip()))
        else:
            print("Warning: label CSV references missing file:", full)
else:
    # Fallback heuristic: infer label from filename keywords (rename your files to include whisper/emotional/neutral for auto-labeling)
    print("Label CSV not found. Inferring labels from filenames (requires 'whisper'/'emotional'/'neutral' in filenames).")
    for f in audio_files:
        fname = os.path.basename(f).lower()
        if "whisper" in fname:
            lab = "whispered"
        elif "emot" in fname or "cry" in fname or "sob" in fname:
            lab = "emotional"
        elif "neutral" in fname:
            lab = "neutral"
        else:
            # If unknown, label as neutral (you should label properly)
            lab = "neutral"
        df.append((f, lab))

print("Training set size (files):", len(df))
if len(df) < 3:
    print("WARNING: Very few training files. For meaningful model you should provide many labeled examples.")

# ------------------ Extract embeddings (this can take a while) ------------------
X = []
y = []
for f, lab in tqdm(df, desc="Extract embeddings"):
    emb = get_embedding(f)
    X.append(emb)
    y.append(lab)
X = np.vstack(X)
y = np.array(y)

# ------------------ Preprocessing & train/test split ------------------
le = LabelEncoder()
y_enc = le.fit_transform(y)

scaler = StandardScaler()
Xs = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(Xs, y_enc, test_size=0.25, random_state=42, stratify=y_enc)

# ------------------ Train classifier ------------------
clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train, y_train)

# Eval
y_pred = clf.predict(X_test)
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ------------------ Save artifacts ------------------
joblib.dump({
    "clf": clf,
    "scaler": scaler,
    "label_encoder": le,
    "wav2vec_model_name": HF_WAV2VEC
}, os.path.join(OUTPUT_DIR, "tone_clf.joblib"))
print("Saved tone_clf.joblib to", OUTPUT_DIR)

# Save a small manifest of training files used
with open(os.path.join(OUTPUT_DIR, "tone_train_manifest.json"), "w") as fh:
    json.dump([{"file": f, "label": l} for f,l in df], fh, indent=2)
print("Training finished.")


Mounted at /content/drive
Device: cuda
Loading Wav2Vec2 feature extractor & model (this may take a moment)...


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Label CSV not found. Inferring labels from filenames (requires 'whisper'/'emotional'/'neutral' in filenames).
Training set size (files): 5


Extract embeddings: 100%|██████████| 5/5 [00:02<00:00,  2.14it/s]



Classification report:
              precision    recall  f1-score   support

     neutral       1.00      1.00      1.00         2

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2

Confusion Matrix:
[[2]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


Saved tone_clf.joblib to /content/drive/MyDrive/Hackthon/INNOV8_outputs
Training finished.


In [ ]:
# Install / ensure packages (run once)
!apt-get update -y
!apt-get install -y ffmpeg -qq
!pip install -q git+https://github.com/openai/whisper.git transformers soundfile librosa scikit-learn joblib pandas


Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
  Installing b

In [ ]:
# Evaluation pipeline (run this Python cell)
import os, glob, re, json, math
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import joblib
import torch
import librosa
import whisper
from transformers import pipeline, Wav2Vec2FeatureExtractor, Wav2Vec2Model

# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ---------------- CONFIG ----------------
EVAL_FOLDER = "/content/drive/MyDrive/Hackthon/INNOV8 3.0/Evaluation set/audio"
ARTIFACTS_DIR = "/content/drive/MyDrive/Hackthon/INNOV8_outputs"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

TONE_ARTIFACT = os.path.join(ARTIFACTS_DIR, "tone_clf.joblib")  # produced by training notebook
WHISPER_MODEL_NAME = "small"   # whisper small model for ASR; change to "base" or "tiny" for speed
TEXT_EMOTION_MODEL = "j-hartmann/emotion-english-distilroberta-base"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ---------------- Load models ----------------
print("Loading whisper...")
whisper_model = whisper.load_model(WHISPER_MODEL_NAME)

if os.path.exists(TONE_ARTIFACT):
    tone_artifacts = joblib.load(TONE_ARTIFACT)
    tone_clf = tone_artifacts["clf"]
    scaler = tone_artifacts["scaler"]
    label_encoder = tone_artifacts["label_encoder"]
    wav2vec_model_name = tone_artifacts.get("wav2vec_model_name", "facebook/wav2vec2-base-960h")
    print("Loaded tone classifier:", TONE_ARTIFACT)
else:
    tone_clf = None
    scaler = None
    label_encoder = None
    wav2vec_model_name = None
    print("WARNING: Tone model not found at", TONE_ARTIFACT, " - tone predictions will use heuristics.")

print("Loading text-emotion model (this may take a bit)...")
emo_pipe = pipeline("text-classification", model=TEXT_EMOTION_MODEL, return_all_scores=True, device=0 if device=="cuda" else -1)

# For embeddings (if clf exists and needs embeddings)
if wav2vec_model_name:
    print("Loading Wav2Vec2 model for embedding extraction:", wav2vec_model_name)
    feat_ext = Wav2Vec2FeatureExtractor.from_pretrained(wav2vec_model_name)
    wv_model = Wav2Vec2Model.from_pretrained(wav2vec_model_name).to(device)
    wv_model.eval()
else:
    feat_ext = None
    wv_model = None

# ---------------- Utility: serializable JSON converter ----------------
def to_serializable(val):
    import numpy as np
    # numpy types -> Python native
    if isinstance(val, (np.bool_,)):
        return bool(val)
    if isinstance(val, (np.integer,)):
        return int(val)
    if isinstance(val, (np.floating,)):
        return float(val)
    if isinstance(val, (np.ndarray,)):
        return val.tolist()
    return str(val)

# ---------------- Audio features (for heuristics) ----------------
def acoustic_features(path, sr=16000):
    y, sr = librosa.load(path, sr=sr)
    rms = float(np.mean(librosa.feature.rms(y=y)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    spec_cent = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    # pitch via librosa.yin (may fail for weak signals)
    try:
        f0 = librosa.yin(y, fmin=50, fmax=600, sr=sr)
        f0_mean = float(np.nanmean(f0))
        f0_std = float(np.nanstd(f0))
    except Exception:
        f0_mean = None
        f0_std = None
    # approximate snr
    amp = np.abs(y) + 1e-9
    snr_approx = float(np.percentile(amp, 90) / (np.percentile(amp, 5) + 1e-9))
    return {"rms": rms, "zcr": zcr, "spec_cent": spec_cent, "f0_mean": f0_mean, "f0_std": f0_std, "snr": snr_approx}

# ---------------- Wav2Vec embedding helper ----------------
def wav2vec_embedding(file_path, sr=16000):
    wav, _ = librosa.load(file_path, sr=sr)
    inputs = feat_ext(wav, sampling_rate=sr, return_tensors="pt", padding=True)
    with torch.no_grad():
        iv = inputs.input_values.to(device)
        outs = wv_model(iv)
        emb = outs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return emb

# ---------------- NLP extractors ----------------
LANGUAGE_KEYWORDS = ["python","java","c++","c#","javascript","js","sql","ruby","go","rust","bash","matlab","r"]
SKILL_KEYWORDS = ["machine learning","deep learning","nlp","data science","devops","docker","kubernetes","testing","aws","azure","gcp","computer vision","django","tensorflow","pytorch"]

def extract_experience_claims(text):
    vals = []
    # find "X years" or "X months"
    for m in re.finditer(r'(\d+(?:\.\d+)?)\s*(years|year|yrs|yr|months|month)', text, flags=re.I):
        num = float(m.group(1))
        unit = m.group(2).lower()
        if "month" in unit:
            num = num / 12.0
        vals.append(round(num,2))
    return vals

def extract_languages(text):
    t = text.lower()
    found = [lang for lang in LANGUAGE_KEYWORDS if (" " + lang + " " in (" " + t + " ")) or (t.startswith(lang+" ")) or (t.endswith(" "+lang))]
    # also check words directly
    for lang in LANGUAGE_KEYWORDS:
        if re.search(r'\b' + re.escape(lang) + r'\b', t):
            if lang not in found:
                found.append(lang)
    return list(set(found))

def extract_mastery(text):
    t = text.lower()
    mapping = {
        "expert": ["expert", "mastered", "master", "proficient", "proficiency"],
        "advanced": ["advanced", "strong experience", "strong"],
        "intermediate": ["intermediate", "competent", "mid-level"],
        "beginner": ["beginner", "learning", "novice", "intern"]
    }
    out = []
    for lab, kws in mapping.items():
        for kw in kws:
            if kw in t:
                out.append(lab)
                break
    return out

def extract_leadership_and_team(text):
    t = text.lower()
    leadership = False
    team_hint = None
    if re.search(r'\bled\b|\blead(?:er|ing)?\b|\bmanaged\b|\bheaded\b|\bteam lead\b|\blead engineer\b|\blead dev\b', t):
        leadership = True
    if re.search(r'\b(alone|solo|by myself|individual contributor|intern)\b', t):
        team_hint = "individual contributor"
    if re.search(r'\b(team|we|us|teammates|our team|led a team|team of)\b', t):
        team_hint = "team member"
    return leadership, team_hint

def extract_skills(text):
    t = text.lower()
    s = []
    for kw in SKILL_KEYWORDS:
        if kw in t:
            s.append(kw)
    return list(set(s))

# ------------------ Group audio files by subject id ------------------
def subject_id_from_filename(fn):
    # attempt to capture prefix before last "_<index>" pattern; e.g. titan_2023_1.mp3 -> titan_2023
    name = os.path.basename(fn)
    name_wo_ext = os.path.splitext(name)[0]
    m = re.match(r'^(.+?)_(\d+)$', name_wo_ext)
    if m:
        return m.group(1)
    # fallback: remove trailing single digit group
    parts = name_wo_ext.split('_')
    if len(parts) > 1 and parts[-1].isdigit():
        return '_'.join(parts[:-1])
    # else return whole name
    return name_wo_ext

audio_files = sorted(glob.glob(os.path.join(EVAL_FOLDER, "*.mp3")) + glob.glob(os.path.join(EVAL_FOLDER, "*.wav")))
if not audio_files:
    raise SystemExit(f"No audio files found in {EVAL_FOLDER}")

subjects = defaultdict(list)
for f in audio_files:
    sid = subject_id_from_filename(f)
    subjects[sid].append(f)

# sort sessions by trailing index (if present)
def sort_sessions(filelist):
    def idx_key(fp):
        n = os.path.splitext(os.path.basename(fp))[0]
        m = re.search(r'_(\d+)$', n)
        if m:
            return int(m.group(1))
        # else try digits inside name
        m2 = re.findall(r'(\d+)', n)
        return int(m2[-1]) if m2 else 0
    return sorted(filelist, key=idx_key)

for k in list(subjects.keys()):
    subjects[k] = sort_sessions(subjects[k])

# ------------------ Per-subject processing ------------------
def analyze_subject(subject_id, file_list):
    sessions_info = []
    # gather acoustic stats so we can mark whispered by relative rms
    rms_vals = []
    for f in file_list:
        ac = acoustic_features(f)
        rms_vals.append(ac["rms"])
    rms_thresh = np.percentile(rms_vals, 30) if len(rms_vals)>1 else None

    for f in file_list:
        print(f"Processing: {subject_id} / {os.path.basename(f)}")
        # acoustic
        ac = acoustic_features(f)
        # tone prediction: either via trained clf (embeddings->clf) OR heuristics
        tone = None
        tone_conf = None
        if tone_clf is not None and feat_ext is not None and wv_model is not None:
            try:
                emb = wav2vec_embedding(f)
                embs = scaler.transform([emb])
                pred = tone_clf.predict(embs)[0]
                tone = label_encoder.inverse_transform([pred])[0]
            except Exception as e:
                # fallback to heuristic
                tone = None

        if tone is None:
            # heuristic: whispered => low RMS and low f0_std; emotional => presence of "sob"/"cry" in ASR or high emotion text
            if rms_thresh is not None and ac["rms"] < rms_thresh:
                tone = "whispered"
            else:
                tone = "neutral"

        # ASR transcription (Whisper)
        try:
            tr = whisper_model.transcribe(f, language="en")
            transcript = tr.get("text","").strip()
        except Exception as e:
            transcript = ""
        # text-emotion
        emotion_label = None
        emotion_score = 0.0
        try:
            emo_scores = emo_pipe(transcript[:1000])[0]
            emo_sorted = sorted(emo_scores, key=lambda x: x['score'], reverse=True)
            if emo_sorted:
                emotion_label = emo_sorted[0]['label']
                emotion_score = float(emo_sorted[0]['score'])
        except Exception:
            emotion_label = None

        # heuristic emotional override: transcripts containing 'sob' or 'cry' or 'sobbing'
        if re.search(r'\b(sob|sobb|cry|crying|i am crying)\b', transcript.lower()):
            emotion_label = "sadness"
            emotion_score = max(emotion_score, 0.9)

        # extract claims
        exp_claims = extract_experience_claims(transcript)
        langs = extract_languages(transcript)
        mastery = extract_mastery(transcript)
        leadership_flag, team_hint = extract_leadership_and_team(transcript)
        skills = extract_skills(transcript)

        # mark heuristics
        is_whispered_heu = (ac["rms"] < rms_thresh) if rms_thresh is not None else False
        is_emotional_heu = (emotion_label is not None and emotion_score > 0.45) or bool(re.search(r'\b(sob|cry|sobb)\b', transcript.lower()))

        sessions_info.append({
            "file": os.path.basename(f),
            "session_path": f,
            "tone": tone,
            "tone_confidence": None,
            "acoustic": ac,
            "is_whispered_heu": bool(is_whispered_heu),
            "transcript": transcript,
            "emotion": emotion_label,
            "emotion_score": emotion_score,
            "is_emotional_heu": bool(is_emotional_heu),
            "experience_claims": exp_claims,
            "programming_languages": langs,
            "skill_mastery_claims": mastery,
            "leadership_flag": bool(leadership_flag),
            "team_hint": team_hint,
            "skills": skills
        })
    # aggregate subject-level truth + deception patterns
    agg = aggregate_subject(sessions_info, subject_id)
    return agg

# ------------------ Aggregation & deception detection ------------------
def aggregate_subject(sessions, subject_id):
    # flatten claims
    exp_vals = [v for s in sessions for v in s["experience_claims"]]
    languages = [l for s in sessions for l in s["programming_languages"]]
    mastery = [m for s in sessions for m in s["skill_mastery_claims"]]
    leadership_flags = [s["leadership_flag"] for s in sessions]
    team_hints = [s["team_hint"] for s in sessions if s["team_hint"]]
    skills = list({kw for s in sessions for kw in s["skills"]})

    # decide programming_experience
    if exp_vals:
        med = float(np.median(exp_vals))
        # build a small range if values differ
        unique_sorted = sorted(set(exp_vals))
        if len(unique_sorted) == 1:
            exp_str = f"{int(round(unique_sorted[0]))} years" if unique_sorted[0].is_integer() else f"{unique_sorted[0]:.1f} years"
        else:
            lo = math.floor(min(unique_sorted))
            hi = math.ceil(max(unique_sorted))
            exp_str = f"{lo}-{hi} years"
    else:
        exp_str = "unknown"

    # programming language (most frequent)
    lang_final = Counter(languages).most_common(1)[0][0] if languages else "unknown"

    # skill_mastery: majority
    skill_mastery = Counter(mastery).most_common(1)[0][0] if mastery else "unknown"

    # leadership summary: detect contradictions
    leadership_summary = "unknown"
    if any(leadership_flags) and any(h == "individual contributor" for h in team_hints):
        leadership_summary = "fabricated"
    elif any(leadership_flags):
        leadership_summary = "true"
    elif any(h == "individual contributor" for h in team_hints):
        leadership_summary = "false"

    team_experience = Counter(team_hints).most_common(1)[0][0] if team_hints else "unknown"

    # deception patterns
    deception = []
    # experience inflation: if two or more different claims and difference > 1 year
    if len(set(exp_vals)) >= 2:
        if max(exp_vals) - min(exp_vals) >= 1.0:
            contradictory_claims = sorted({f"{v} years" if float(v).is_integer() else f"{v:.1f} years" for v in exp_vals})
            deception.append({"lie_type": "experience_inflation", "contradictory_claims": contradictory_claims})

    # leadership contradictions
    if len(set(leadership_flags)) > 1 or (any(leadership_flags) and any(h == "individual contributor" for h in team_hints)):
        contra = []
        if any(leadership_flags):
            contra.append("claimed leadership")
        if any(h == "individual contributor" for h in team_hints):
            contra.append("claimed solo/individual")
        deception.append({"lie_type": "leadership_claims", "contradictory_claims": contra})

    # fabricated strong claims: detect statements like "architected the entire solution" vs "I was an intern"
    strong_claims = []
    for s in sessions:
        t = s["transcript"].lower()
        if re.search(r'\barchitect(ed|ure)? the (entire|whole)\b|\bbuilt (the )?entire\b|\bfrom the very first line\b|\bi am a principal\b|\bi am a lead\b|\bi was the lead\b', t):
            strong_claims.append(s["file"])
    if strong_claims and any("intern" in s["transcript"].lower() for s in sessions):
        deception.append({"lie_type": "fabricated_role", "contradictory_claims": ["strong claims vs intern statements"]})

    result = {
        "shadow_id": subject_id,
        "revealed_truth": {
            "programming_experience": exp_str,
            "programming_language": lang_final,
            "skill_mastery": skill_mastery,
            "leadership_claims": leadership_summary,
            "team_experience": team_experience,
            "skills and other keywords": skills
        },
        "deception_patterns": deception,
        "sessions": sessions
    }
    return result

# ----------------- Run for all subjects -----------------
all_results = []
for sid, flist in subjects.items():
    print("----- SUBJECT:", sid, " sessions:", len(flist))
    res = analyze_subject(sid, flist)
    all_results.append(res)
    # write per-subject file
    out_path = os.path.join(ARTIFACTS_DIR, f"{sid}_final.json")
    with open(out_path, "w") as fh:
        json.dump(res, fh, indent=2, default=to_serializable)
    print("Wrote:", out_path)

# Save a combined JSON
combined_path = os.path.join(ARTIFACTS_DIR, "audio_final.json")
with open(combined_path, "w") as fh:
    json.dump(all_results, fh, indent=2, default=to_serializable)

print("✅ All done. Outputs saved to:", ARTIFACTS_DIR)
print("Combined JSON:", combined_path)


Mounted at /content/drive
Device: cuda
Loading whisper...
Loaded tone classifier: /content/drive/MyDrive/Hackthon/INNOV8_outputs/tone_clf.joblib
Loading text-emotion model (this may take a bit)...


Device set to use cuda:0
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


Loading Wav2Vec2 model for embedding extraction: facebook/wav2vec2-base-960h


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


----- SUBJECT: atlas_2025  sessions: 5
Processing: atlas_2025 / atlas_2025_1.mp3
Processing: atlas_2025 / atlas_2025_2.mp3
Processing: atlas_2025 / atlas_2025_3.mp3
Processing: atlas_2025 / atlas_2025_4.mp3
Processing: atlas_2025 / atlas_2025_5.mp3
Wrote: /content/drive/MyDrive/Hackthon/INNOV8_outputs/atlas_2025_final.json
----- SUBJECT: crius_2025  sessions: 1
Processing: crius_2025 / crius_2025_5.mp3
Wrote: /content/drive/MyDrive/Hackthon/INNOV8_outputs/crius_2025_final.json
----- SUBJECT: eos_2023  sessions: 5
Processing: eos_2023 / eos_2023_1.mp3
Processing: eos_2023 / eos_2023_2.mp3
Processing: eos_2023 / eos_2023_3.mp3
Processing: eos_2023 / eos_2023_4.mp3
Processing: eos_2023 / eos_2023_5.mp3
Wrote: /content/drive/MyDrive/Hackthon/INNOV8_outputs/eos_2023_final.json
----- SUBJECT: hyperion_2022  sessions: 5
Processing: hyperion_2022 / hyperion_2022_1.mp3
Processing: hyperion_2022 / hyperion_2022_2.mp3
Processing: hyperion_2022 / hyperion_2022_3.mp3
Processing: hyperion_2022 / hyp